# StateGen — DS-1000 Experiments (Colab)

Runs all four code generation methods (Direct Generation, Self-Planning, Self-Debugging, StateGen) on the DS-1000 benchmark (100 Pandas + 100 NumPy) with execution-based evaluation.

**Prerequisites:**
- A `.env` file with `DEEPSEEK_API_KEY` (or `TOGETHER_API_KEY` if using Together.ai)
- No GPU required (CPU runtime is sufficient)

**Expected runtime:** ~2–3 hours for 200 tasks × 4 methods

In [1]:
%cd /content
!rm -rf StateGen
!git clone https://github.com/Joshh99/StateGen.git
%cd StateGen


/content
Cloning into 'StateGen'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 211 (delta 87), reused 186 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 5.83 MiB | 16.15 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/StateGen


In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install litellm python-dotenv datasets sentence-transformers

In [ ]:
from google.colab import files
uploaded = files.upload()
import shutil
shutil.move(list(uploaded.keys())[0], ".env")

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

for var in ["DEEPSEEK_API_KEY", "AZURE_API_KEY", "GOOGLE_API_KEY"]:
    val = os.getenv(var)
    print(f"{var}: {'SET' if val else 'MISSING'}")

## Configuration
Edit the variables below before running experiments.

In [ ]:
RUN = "gemini_flash"  # Options: "azure_gpt4o", "gemini_pro", "gemini_flash"

CONFIG = {
    "azure_gpt4o": {
        "provider": "azure",
        "model": "gpt-4o-stategen",
        "results_dir": "results/ds1000_gpt4o",
    },
    "gemini_pro": {
        "provider": "gemini",
        "model": "gemini/gemini-2.5-pro",
        "results_dir": "results/ds1000_gemini_pro",
    },
    "gemini_flash": {
        "provider": "gemini",
        "model": "gemini/gemini-2.5-flash",
        "results_dir": "results/ds1000_gemini_flash",
    },
}

PROVIDER = CONFIG[RUN]["provider"]
MODEL = CONFIG[RUN]["model"]
RESULTS_DIR = CONFIG[RUN]["results_dir"]
MAX_TASKS = 200

print(f"Running: {RUN} -> {PROVIDER}/{MODEL}")
print(f"Results: {RESULTS_DIR}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP = "/content/drive/MyDrive/StateGen_Results"
import os; os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f"Drive backup: {DRIVE_BACKUP}")

In [ ]:
!python experiments/run_ds1000.py \
  --method direct_gen \
  --max_tasks 2 \
  --dry_run

In [ ]:
# Run this first to sanity check before full 200-task run
import subprocess
result = subprocess.run([
    "python", "experiments/run_ds1000.py",
    "--method", "direct_gen",
    "--max_tasks", "10",
    "--provider", PROVIDER,
    "--model", MODEL,
    "--results_dir", RESULTS_DIR
], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## Full Experiment Run
Check pilot results above before proceeding.
Expected cost: ~$3-5 on Together.ai for full 200 tasks x 4 methods.

In [ ]:
# Run sequentially to avoid rate limits
import subprocess
import shutil
METHODS = ["direct_gen", "self_planning", "self_debugging", "stategen"]
for method in METHODS:
    print(f"\n{'='*50}\nRunning: {method}\n{'='*50}")
    result = subprocess.run([
        "python", "experiments/run_ds1000.py",
        "--method", method,
        "--max_tasks", str(MAX_TASKS),
        "--provider", PROVIDER,
        "--model", MODEL,
        "--results_dir", RESULTS_DIR
    ], capture_output=True, text=True)
    print(result.stdout[-3000:])  # last 3000 chars to avoid cell overflow
    if result.returncode != 0:
        print("ERROR:", result.stderr[-1000:])
        break  # stop on first failure
    shutil.copytree(RESULTS_DIR, DRIVE_BACKUP, dirs_exist_ok=True)
    print(f"Backed up to Drive after {method}")

In [ ]:
import json, os
metrics_path = f"{RESULTS_DIR}/metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    print(json.dumps(m, indent=2))
else:
    print("No metrics file yet -- run experiments first")

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("ds1000_results", "zip", RESULTS_DIR)
files.download("ds1000_results.zip")